# Lesson 04 - 缺失值、重複值與型態轉換

1. 真實資料常會有缺失值、重複值或型態不適合分析的問題。
2. 這一章練習基本資料清理。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR_CANDIDATES = [
    Path("../data/raw"),
    Path("data/raw"),
    Path("/content/Py_dataAna/code/data/raw"),
    Path("/content/code/data/raw"),
]

DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if (path / "orders.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "找不到 CSV 資料。請確認 code/data/raw/ 內有 orders.csv 等資料檔，"
        "在 Colab 可先上傳整個專案資料夾或掛載 Google Drive。"
    )

print("Using data folder:", DATA_DIR.resolve())

Using data folder: E:\py_20260620\data\raw


In [2]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
events = pd.read_csv(DATA_DIR / "events.csv")
ab_assignments = pd.read_csv(DATA_DIR / "ab_assignments.csv")

tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "sessions": sessions,
    "events": events,
    "ab_assignments": ab_assignments,
}
pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in tables.items()]
)

,table,rows,columns
0,customers,2500,5
1,products,60,3
2,orders,22000,5
3,order_items,39627,5
4,sessions,70000,7
5,events,232067,6
6,ab_assignments,2500,3


## 檢查缺失值

In [3]:
sessions.isna().sum().sort_values(ascending=False)

session_id          0
customer_id         0
session_start       0
device              0
traffic_source      0
campaign            0
experiment_group    0
dtype: int64

## 檢查重複資料

In [4]:
print("sessions duplicated rows:", sessions.duplicated().sum())
print("orders duplicated order_id:", orders["order_id"].duplicated().sum())

sessions duplicated rows: 0
orders duplicated order_id: 0


## 日期型態轉換

In [8]:
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
sessions = sessions.copy()
print(sessions["session_start"].dtype)#查詢欄位類型
sessions["session_start"] = pd.to_datetime(sessions["session_start"])
sessions[["session_id", "session_start"]].head()
print(sessions["session_start"].dtype)#查詢欄位類型

str
datetime64[us]


## 從日期時間拆出新欄位

In [9]:
sessions["year"] = sessions["session_start"].dt.year # 年份

sessions["session_hour"] = sessions["session_start"].dt.hour
sessions["weekday"] = sessions["session_start"].dt.day_name()#星期
sessions["is_weekend"] = sessions["session_start"].dt.dayofweek >= 5

sessions[["session_start","year" ,"session_hour", "weekday", "is_weekend"]].head()

,session_start,year,session_hour,weekday,is_weekend
0,2025-08-14 11:02:00,2025,11,Thursday,False
1,2024-02-29 11:56:00,2024,11,Thursday,False
2,2024-11-13 16:23:00,2024,16,Wednesday,False
3,2024-04-22 16:15:00,2024,16,Monday,False
4,2024-06-15 08:39:00,2024,8,Saturday,True


### 日期篩選

In [11]:
df = pd.read_csv(DATA_DIR / "sessions.csv")
df['session_start'] = pd.to_datetime(df['session_start'])
#篩選 year=2025
y2025 = df.loc[df['session_start'].dt.year == 2025]
len(y2025)

34963

## 小練習

換成 `orders['order_date']` 做日期轉換，並拆出月份。